# IEC104 — Parallel Binary Classifiers + SHAP

**Dataset**: SANDI-2024 (Substation Anomaly Network Data for Intrusion detection 2024)  
**Paper**: Gutiérrez Mlot et al., *Data in Brief* 57 (2024) 111153  

## Approach

Train 7 independent binary classifiers — one per attack type vs normal traffic.  
Each model answers one focused question: *"Is this [attack X] or normal traffic?"*

**Why not the multiclass model?**
- The benchmark's MCC=0.9996 hides per-class difficulty (mitmattack has only 26 samples total)
- SHAP on an 8-class model is hard to interpret
- Binary models expose true detection difficulty per attack type
- Binary SHAP = one clean answer per model: *what makes this attack distinguishable from normal traffic?*

**Key design choices**:
- `class_weight='balanced'` — handles both attack-dominant (dosattack 1194:1) and normal-dominant (mitmattack 0.1:1) imbalance without SMOTE
- sklearn ExtraTrees directly — faster than 7 PyCaret setups; ExtraTrees won the benchmark comparison
- StandardScaler (zscore) — consistent with the benchmark notebook
- Stratified 70/30 split, seed=123 — same as benchmark for fair comparison

## Sections
1. **Training loop** — 7 binary ExtraTrees models, evaluation + SHAP per model
2. **Summary table** — all models' precision / recall / F1 / MCC
3. **Cross-model heatmap** — which features matter across attack types

In [1]:
# Run once to install SHAP, then restart kernel if needed
%pip install shap --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import warnings
import pickle
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    matthews_corrcoef, cohen_kappa_score
)

pd.set_option('display.max_columns', 10)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Paths (same as iec104_benchmark.ipynb) ─────────────────────────────────
DATA_ROOT     = '/Users/metis/Library/Mobile Documents/com~apple~CloudDocs/UT-iC/01 Project/15487636'
IEC104_DIR    = os.path.join(DATA_ROOT, 'processed-iec104', 'iec104')
HEADERS_FILE  = os.path.join(IEC104_DIR, 'headers_iec104.txt')
NOTEBOOKS_DIR = os.getcwd()

print('IEC104 dir   :', IEC104_DIR)
print('Notebooks dir:', NOTEBOOKS_DIR)
print('Data exists? :', os.path.exists(IEC104_DIR))

In [ ]:
# ── Load dataset (same logic as iec104_benchmark.ipynb) ───────────────────
def read_headers(filename):
    with open(filename, 'r') as f:
        data = f.read()
    values = data.split(',\n')
    values[-1] = values[-1].rstrip('\n')
    return values

cols = read_headers(HEADERS_FILE)
print(f'Feature columns: {len(cols)}')

csv_files = []
for root, dirs, files in os.walk(IEC104_DIR):
    for name in sorted(files):
        if name.endswith('.csv'):
            csv_files.append(os.path.join(root, name))

frames = []
for path in csv_files:
    frames.append(pd.read_csv(path, usecols=cols))

df = pd.concat(frames, ignore_index=True)
df.dropna(axis=1, inplace=True)
df = df[~df.isin([np.nan, np.inf, -np.inf]).any(axis=1)]

print(f'Loaded: {len(df):,} rows x {df.shape[1]} columns')
print('Class counts:')
print(df['Label'].value_counts().to_string())

---
## Section 1 — Binary Classifier Training Loop

For each of the 7 attack classes:
1. Subset data to `[attack_class rows] + [attackfree rows]`
2. Binary encode: `1 = attack`, `0 = normal`
3. Stratified 70/30 split → StandardScaler → ExtraTrees(class_weight='balanced')
4. Evaluate: classification report + confusion matrix
5. SHAP: beeswarm + bar plots, store top-10 features
6. Save model pkl

In [ ]:
ATTACK_CLASSES = [
    'dosattack',
    'floodattack',
    'fuzzyattack',
    'iec104starvationattack',
    'mitmattack',
    'ntpddosattack',
    'portscanattack',
]

results   = []   # collects per-model metrics
shap_top  = {}   # attack_class -> {feature: mean_abs_shap} for top-10

print('Attack classes to process:', ATTACK_CLASSES)

In [ ]:
SEP = '=' * 65

for attack_class in ATTACK_CLASSES:

    print()
    print(SEP)
    print('MODEL:', attack_class, 'vs attackfree')
    print(SEP)

    # ── 1. Subset ──────────────────────────────────────────────────────────
    mask   = df['Label'].isin([attack_class, 'attackfree'])
    df_bin = df[mask].copy()
    y      = (df_bin['Label'] == attack_class).astype(int)
    X      = df_bin.drop(columns=['Label'])
    feature_cols = list(X.columns)

    n_attack = int(y.sum())
    n_normal = int((y == 0).sum())
    print(f'  Attack samples : {n_attack:,}')
    print(f'  Normal samples : {n_normal:,}')

    # ── 2. Stratified 70/30 split ──────────────────────────────────────────
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=123
    )

    # ── 3. Scale (zscore, fit on train only) ───────────────────────────────
    scaler    = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    # ── 4. Train ───────────────────────────────────────────────────────────
    clf = ExtraTreesClassifier(
        class_weight='balanced',
        n_estimators=100,
        random_state=123,
        n_jobs=-1
    )
    clf.fit(X_train_s, y_train)
    print('  Trained.')

    # ── 5. Evaluate ────────────────────────────────────────────────────────
    y_pred = clf.predict(X_test_s)
    mcc    = matthews_corrcoef(y_test, y_pred)
    kappa  = cohen_kappa_score(y_test, y_pred)

    print()
    print(classification_report(
        y_test, y_pred,
        target_names=['attackfree', attack_class],
        zero_division=0
    ))
    print(f'  MCC: {mcc:.4f}  |  Kappa: {kappa:.4f}')

    # Confusion matrix
    cm  = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Greens',
        xticklabels=['attackfree', attack_class],
        yticklabels=['attackfree', attack_class],
        ax=ax
    )
    ax.set_title('Binary CM: ' + attack_class + ' vs attackfree')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.tight_layout()
    cm_path = os.path.join(NOTEBOOKS_DIR, 'binary_cm_' + attack_class + '.png')
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

    # ── 6. SHAP ────────────────────────────────────────────────────────────
    # Sample up to 500 test rows — keeps SHAP tractable for large classes
    print('  Computing SHAP values (sample up to 500 rows)...')
    shap_n  = min(500, len(X_test_s))
    rng     = np.random.RandomState(123)
    idx     = rng.choice(len(X_test_s), shap_n, replace=False)
    X_shap  = X_test_s[idx]

    explainer = shap.TreeExplainer(clf)
    sv        = explainer.shap_values(X_shap)
    sv_attack = sv[1]   # class 1 = attack

    # Beeswarm (shows direction + magnitude per feature per sample)
    shap.summary_plot(
        sv_attack, X_shap,
        feature_names=feature_cols,
        max_display=20,
        show=False
    )
    plt.title('SHAP Beeswarm: ' + attack_class + ' vs attackfree (top 20 features)')
    plt.tight_layout()
    bee_path = os.path.join(NOTEBOOKS_DIR, 'shap_beeswarm_' + attack_class + '.png')
    plt.savefig(bee_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

    # Bar (mean |SHAP| — global feature importance)
    shap.summary_plot(
        sv_attack, X_shap,
        feature_names=feature_cols,
        max_display=20,
        plot_type='bar',
        show=False
    )
    plt.title('SHAP Mean |value|: ' + attack_class + ' vs attackfree')
    plt.tight_layout()
    bar_path = os.path.join(NOTEBOOKS_DIR, 'shap_bar_' + attack_class + '.png')
    plt.savefig(bar_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

    # Store top-10 features by mean |SHAP|
    mean_abs = np.abs(sv_attack).mean(axis=0)
    top_idx  = np.argsort(mean_abs)[::-1][:10]
    shap_top[attack_class] = {feature_cols[i]: float(mean_abs[i]) for i in top_idx}

    # ── 7. Save model ──────────────────────────────────────────────────────
    pkl_path = os.path.join(NOTEBOOKS_DIR, 'binary_' + attack_class + '.pkl')
    with open(pkl_path, 'wb') as fh:
        pickle.dump(
            {'scaler': scaler, 'clf': clf,
             'attack_class': attack_class,
             'feature_cols': feature_cols},
            fh
        )
    print('  Model saved:', pkl_path)

    # ── 8. Collect results ─────────────────────────────────────────────────
    report_dict = classification_report(
        y_test, y_pred,
        target_names=['attackfree', attack_class],
        zero_division=0,
        output_dict=True
    )
    r = report_dict[attack_class]
    results.append({
        'attack_class':    attack_class,
        'n_train_attack':  int((y_train == 1).sum()),
        'n_test_attack':   int((y_test  == 1).sum()),
        'precision':       round(r['precision'], 4),
        'recall':          round(r['recall'],    4),
        'f1':              round(r['f1-score'],  4),
        'MCC':             round(mcc,            4),
        'Kappa':           round(kappa,          4),
    })

print()
print('All 7 binary classifiers complete.')

---
## Section 2 — Summary Table

All 7 binary models compared by attack class.  
**Expected finding**: mitmattack and floodattack will show lower scores than the multiclass benchmark — this is true difficulty that was hidden before.

In [ ]:
summary_df = pd.DataFrame(results).set_index('attack_class')
summary_df = summary_df.sort_values('MCC', ascending=False)

print('=== Binary Classifier Summary ===')
print(summary_df.to_string())

csv_path = os.path.join(NOTEBOOKS_DIR, 'binary_summary.csv')
summary_df.to_csv(csv_path)
print('\nSaved:', csv_path)

---
## Section 3 — Cross-Model Feature Importance Heatmap

Rows = union of top-10 features across all 7 models (by mean |SHAP|)  
Columns = attack classes  
Values = mean |SHAP| for that feature in that model

**What to look for**:
- Features that are bright across many columns → universally important anomaly signals
- Features that are bright in only one column → attack-type-specific signatures
- Empty cells (0.0) → feature was not in the top-10 for that attack

In [ ]:
# Build union of top-10 features across all models
all_features = set()
for feat_dict in shap_top.values():
    all_features.update(feat_dict.keys())
all_features = sorted(all_features)

heatmap_data = pd.DataFrame(0.0, index=all_features, columns=ATTACK_CLASSES)

for ac, feat_dict in shap_top.items():
    for feat, val in feat_dict.items():
        heatmap_data.loc[feat, ac] = val

# Sort rows by total importance across all attack types
heatmap_data['_total'] = heatmap_data.sum(axis=1)
heatmap_data = heatmap_data.sort_values('_total', ascending=False).drop(columns='_total')

fig, ax = plt.subplots(figsize=(14, max(7, len(heatmap_data) * 0.45)))
sns.heatmap(
    heatmap_data,
    cmap='YlOrRd',
    annot=True,
    fmt='.3f',
    linewidths=0.4,
    ax=ax
)
ax.set_title('Mean |SHAP| — Top Features per Attack Type (binary classifiers)')
ax.set_xlabel('Attack Class')
ax.set_ylabel('Feature')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()

heatmap_path = os.path.join(NOTEBOOKS_DIR, 'shap_crossmodel_heatmap.png')
plt.savefig(heatmap_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', heatmap_path)

---
## Summary

This notebook trained 7 parallel binary classifiers (ExtraTreesClassifier, class_weight='balanced') and applied SHAP TreeExplainer to each.

**Outputs saved**:
- `binary_{attack_class}.pkl` — 7 model files (scaler + classifier + feature list)
- `binary_cm_{attack_class}.png` — 7 confusion matrix heatmaps
- `shap_beeswarm_{attack_class}.png` — 7 SHAP beeswarm plots (top 20 features)
- `shap_bar_{attack_class}.png` — 7 SHAP bar plots (mean |SHAP|)
- `shap_crossmodel_heatmap.png` — cross-model feature importance comparison
- `binary_summary.csv` — summary metrics table

### Next steps
- Interpret SHAP plots per attack: which CICFlowMeter features drive each detection?
- Cross-model heatmap: identify universal anomaly signals vs class-specific fingerprints
- Consider comparing ExtraTrees SHAP against a linear model (LogisticRegression) SHAP